# Preliminary stats for the extracted beliefs (7-day windows)

In [ ]:
import json
import sys
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, "../restart_MrP_pipeline")
from belief_recodes_7d import n_categories, recode_value

DATA_DIR = "../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data"
INPUT_CSV = f"{DATA_DIR}/feature_extraction/all_results_7d.csv"
USER_DIR = f"{DATA_DIR}/feature_extraction/users_7d"
FEATURES_JSON = f"{DATA_DIR}/updated_belief_features.json"

# chart colors: one series, so one hue; gray for context
BAR = "#2a78d6"
BAR_DARK = "#104281"
MUTED = "#9a9992"
INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
GRID = "#e4e3df"
SURFACE = "#fcfcfb"

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 9, "text.color": INK, "axes.labelcolor": INK_SECONDARY,
    "xtick.color": INK_SECONDARY, "ytick.color": INK_SECONDARY,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlelocation": "left", "axes.titlesize": 9, "axes.titlecolor": INK,
    "xtick.major.size": 0, "ytick.major.size": 0,
})
pd.set_option("display.max_rows", 100, "display.max_columns", 30, "display.width", 200, "display.max_colwidth", 60)

## 1. Load

In [ ]:
with open(FEATURES_JSON, "r", encoding="utf-8") as f:
    belief_features = json.load(f)["beliefs"]

# short names for tables and charts -- full wording is in updated_belief_features.json
TOPICS = {
    "LETIN1A": "Number of immigrants", "IMMCRIME1": "Immigrants increase crime", "IMMJOBS1": "Immigrants take jobs",
    "IMMFATE": "Undocumented immigrants: what to do", "DIVLAW": "Divorce easier / harder", "ABANY": "Abortion for any reason",
    "MARSAME": "Same-sex marriage", "FEPOLV": "Men better suited to politics", "FEFAM": "Man works, woman runs home",
    "V242353x": "Spending on health insurance help", "V242357x": "Vaccine benefits vs risks", "COURTS": "Courts too harsh / lenient",
    "CC24_443d": "State law-enforcement spending", "GRASS": "Legalize marijuana", "GUNLAW": "Police permit to buy a gun",
    "CC24_440a": "White people have advantages", "AFFRMACT": "Preferential hiring of Black people",
    "CC24_444a": "Ban gender surgery for minors", "SATDEMOC": "Satisfaction with democracy", "BIASNEWS": "Media coverage is biased",
    "SMNEWS": "Reliability of social-media news", "CC24_326d": "Increase fossil-fuel production",
    "TEMPGEN1": "Danger of climate warming", "V241409x": "Side with Israel / Palestine", "V241403x": "Military aid to Israel",
    "V241400x": "Weapons for Ukraine", "CONLABORY": "Confidence in labor unions", "V241294x": "National economy, past year",
    "V241137x": "Biden approval", "V241129x": "Congress approval", "V241133x": "Supreme Court approval",
    "V241229": "Trust the federal government", "WRKWAYUP": "Work way up without favors",
    "V242301": "Slavery makes moving up hard", "V241608": "Women seek power over men", "V242331x": "Ban assault-style rifles",
    "V242328x": "Background checks, private sales", "V241395x": "Border wall with Mexico", "CAPPUN": "Death penalty",
    "EQWLTH": "Government reduce income gap",
}
assert set(TOPICS) == {f["id"] for f in belief_features}

ids = [f["id"] for f in belief_features]
usecols = ["id", "time_window_start", "username", "is_empty_window", "Belief_Skip_Reason"]
usecols += [f"Belief_{i}_Symbol" for i in ids] + [f"Belief_{i}_Speculation" for i in ids]
raw = pd.read_csv(INPUT_CSV, usecols=usecols, encoding="utf-8-sig", low_memory=False)

nonempty = raw[~raw["is_empty_window"]]
answered = nonempty[nonempty["Belief_Skip_Reason"].isna()].copy()
print(f"{len(raw):,} user-windows from {raw['id'].nunique()} users")
print(f"{len(nonempty):,} non-empty windows ({len(nonempty) / len(raw):.1%})")
print(f"{len(answered):,} with belief answers, from {answered['id'].nunique()} users "
      f"({len(nonempty) - len(answered)} skipped as too long)")
print(f"Windows with answers per user: median {answered.groupby('id').size().median():.0f}, "
      f"max {answered.groupby('id').size().max()}")

In [ ]:
# recoded answers: one column per question, named by id
values = pd.DataFrame({
    f["id"]: answered[f"Belief_{f['id']}_Symbol"].map(lambda v: recode_value(f["id"], v)).astype(float)
    for f in belief_features
})
speculation = pd.DataFrame({f["id"]: answered[f"Belief_{f['id']}_Speculation"].astype(float) for f in belief_features})

def recoded_labels(feature):
    # option text per recoded value; options merged by a recode are joined with " / "
    labels = {}
    for position, text in enumerate(feature["options"], start=1):
        labels.setdefault(recode_value(feature["id"], position), []).append(text)
    return {value: " / ".join(texts) for value, texts in sorted(labels.items())}

option_labels = {f["id"]: recoded_labels(f) for f in belief_features}

## 2. Mean, median and spread by question

`scaled_mean` puts the mean on 0-1 (0 = first option, 1 = last option) so questions with different numbers of options can sit on one axis. `user_mean` averages each user's own mean, so every user counts once. Listed in the order of `updated_belief_features.json`.

In [ ]:
rows = []
user_means = values.groupby(answered["id"]).mean()
for f in belief_features:
    key = f["id"]
    v = values[key]
    k = n_categories(key, len(f["options"]))
    shares = v.value_counts(normalize=True)
    rows.append({
        "id": key, "topic": TOPICS[key], "K": k, "n": int(v.notna().sum()),
        "mean": v.mean(), "median": v.median(), "sd": v.std(),
        "scaled_mean": (v.mean() - 1) / (k - 1),
        "user_mean": user_means[key].mean(),
        "most_common": option_labels[key][int(shares.idxmax())][:45],
        "most_common_share": shares.max(),
        "spec_mean": speculation[key].mean(), "spec_median": speculation[key].median(),
    })
stats = pd.DataFrame(rows)
display(stats.round({"mean": 2, "sd": 2, "scaled_mean": 2, "user_mean": 2, "most_common_share": 2, "spec_mean": 1}))

In [ ]:
# where each question's mean and median fall on its own scale
order = stats.sort_values("scaled_mean")
y = np.arange(len(order))
scaled_median = (order["median"] - 1) / (order["K"] - 1)

fig, ax = plt.subplots(figsize=(8.5, 9.5))
ax.hlines(y, order["scaled_mean"], scaled_median, color=GRID, lw=2, zorder=1)
ax.scatter(scaled_median, y, s=42, marker="D", color=MUTED, edgecolor=SURFACE, lw=1.5, zorder=2, label="Median")
ax.scatter(order["scaled_mean"], y, s=64, color=BAR, edgecolor=SURFACE, lw=1.5, zorder=3, label="Mean")
ax.set_yticks(y, order["topic"])
ax.set_xlim(-0.03, 1.03)
ax.set_ylim(-1, len(order))
ax.set_xticks([0, 0.25, 0.5, 0.75, 1], ["0\nfirst option", "0.25", "0.5", "0.75", "1\nlast option"])
ax.grid(axis="x", color=GRID, lw=0.8)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=2, frameon=False)
ax.set_title("Mean and median position on each question's own scale\n", fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()

## 3. Distribution of answers

Share of answered windows choosing each option, in scale order. The most common option carries its share as a label.

In [ ]:
n_cols, n_rows = 5, 8
fig, axes = plt.subplots(n_rows, n_cols, figsize=(19, 25), sharex=True)
for ax, f in zip(axes.flat, belief_features):
    key = f["id"]
    labels = option_labels[key]
    shares = values[key].value_counts(normalize=True).reindex(labels.keys(), fill_value=0)
    pos = np.arange(len(shares))
    ax.barh(pos, shares.values, height=0.68, color=BAR)
    ax.set_yticks(pos, [textwrap.shorten(t, width=30, placeholder="...") for t in labels.values()], fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.5, 1], ["0%", "50%", "100%"])
    ax.tick_params(axis="x", labelbottom=True)
    ax.grid(axis="x", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    top = int(np.argmax(shares.values))
    ax.text(shares.values[top] + 0.02, top, f"{shares.values[top]:.0%}", va="center", fontsize=8, color=INK)
    row = stats.set_index("id").loc[key]
    ax.set_title(f"{TOPICS[key]}\n", fontweight="bold")
    ax.text(0, 1.02, f"{key}  |  mean {row['mean']:.2f}, median {row['median']:.0f}", transform=ax.transAxes,
            fontsize=7.5, color=INK_SECONDARY)
fig.suptitle("Distribution of extracted answers, by question", x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.985))
plt.show()

In [ ]:
# how often the model picks an explicit neutral option, for questions that have one
NEUTRAL = r"^Neither|^No change|^No difference|^Remain the same|^Stayed about|^Maintain|^Stay as is|^Side with both"
neutral_rows = []
for f in belief_features:
    for value, text in option_labels[f["id"]].items():
        if pd.Series(text).str.contains(NEUTRAL).iloc[0]:
            neutral_rows.append({"topic": TOPICS[f["id"]], "neutral_option": text[:45], "share": (values[f["id"]] == value).mean()})
neutral = pd.DataFrame(neutral_rows).sort_values("share")
print(f"{len(neutral)} questions have a neutral option; median share choosing it: {neutral['share'].median():.1%}")
neutral.round(3)

## 4. Speculation scores

The model's own rating of how little evidence it had: 0-20 explicit evidence, 21-40 strong indirect, 41-60 partial, 61-80 weak, 81-100 no meaningful evidence.

In [ ]:
all_spec = speculation.stack()
by_question = stats.sort_values("spec_mean")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 9), gridspec_kw={"width_ratios": [1, 1]})

ax1.hist(all_spec, bins=np.arange(-2.5, 105, 5), color=BAR, edgecolor=SURFACE, lw=1.5)  # scores are almost all multiples of 5, so center bins on them
for edge in [20, 40, 60, 80]:
    ax1.axvline(edge, color=GRID, lw=1, zorder=0)
median_spec = all_spec.median()
ax1.axvline(median_spec, color=BAR_DARK, lw=2)
ax1.text(median_spec + 1.5, ax1.get_ylim()[1] * 0.96, f"median {median_spec:.0f}", color=INK, va="top")
ax1.set_xlim(0, 100)
ax1.set_xlabel("Speculation score, all belief answers")
ax1.set_ylabel("Answers")
ax1.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax1.set_title("Distribution of speculation scores", fontsize=11, fontweight="bold")

y = np.arange(len(by_question))
ax2.barh(y, by_question["spec_mean"], height=0.68, color=BAR)
ax2.set_yticks(y, by_question["topic"], fontsize=8)
ax2.set_xlim(0, 100)
ax2.set_ylim(-0.8, len(by_question) - 0.2)
ax2.grid(axis="x", color=GRID, lw=0.8)
ax2.set_axisbelow(True)
ax2.spines["left"].set_visible(False)
ax2.set_xlabel("Mean speculation score")
ax2.set_title("Mean speculation by question", fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()

bands = pd.cut(all_spec, [-1, 20, 40, 60, 80, 100], labels=["0-20 explicit", "21-40 strong indirect", "41-60 partial", "61-80 weak", "81-100 none"])
print(f"All belief answers: mean {all_spec.mean():.1f}, median {median_spec:.0f}")
print(bands.value_counts(normalize=True).sort_index().round(3).to_string())

In [ ]:
# is speculation about the question, or about the window as a whole?
window_mean = speculation.mean(axis=1)
print(f"Share of answers on a multiple of 5: {(all_spec % 5 == 0).mean():.1%}")
print(f"Share of all speculation variance that is between windows: {window_mean.var() / all_spec.var():.1%}")
print(f"Typical spread across the 40 questions within one window: median SD {speculation.std(axis=1).median():.1f}")
print(f"Spread of question means across questions: {stats['spec_mean'].min():.1f} to {stats['spec_mean'].max():.1f}")

## 5. Example outputs

Three windows: one at random, plus the windows with the lowest and highest mean speculation. Speculation is the model's self-report, not a check of what the tweets contain. Each shows the input excerpt, the answers with the lowest speculation, and the start of the model's reasoning. Change `N_ANSWERS` to see more of the 40 answers per window.

In [ ]:
N_ANSWERS = 12
window_spec = speculation.mean(axis=1)
examples = {
    "Random window": answered.sample(1, random_state=42).index[0],
    "Lowest mean speculation (model's self-report)": window_spec.idxmin(),
    "Highest mean speculation (model's self-report)": window_spec.idxmax(),
}

def show_example(title, idx, show_raw=False):
    row = answered.loc[idx]
    user = pd.read_csv(f"{USER_DIR}/{row['id']}.csv", encoding="utf-8-sig", low_memory=False)
    full = user[user["time_window_start"] == row["time_window_start"]].iloc[0]

    print("=" * 100)
    print(f"{title}\n@{row['username']}  |  window starting {row['time_window_start'][:10]}  |  mean speculation {window_spec[idx]:.1f}")
    print(f"Bio: {full['description']}\nLocation: {full['location']}")
    tweets = str(full["tweets_struct"])
    print(f"Tweets ({len(tweets):,} characters, first 500):\n{tweets[:500]}\n")

    answers = pd.DataFrame([{
        "question": TOPICS[f["id"]],
        "answer": option_labels[f["id"]][int(values.loc[idx, f["id"]])][:55],
        "speculation": int(speculation.loc[idx, f["id"]]),
    } for f in belief_features]).sort_values("speculation")
    print(f"{N_ANSWERS} of 40 answers, lowest speculation first:")
    print(answers.head(N_ANSWERS).to_string(index=False))

    print(f"\nModel reasoning (first 700 of {len(str(full['Belief_Reasoning'])):,} characters):\n{str(full['Belief_Reasoning'])[:700]}")
    if show_raw:
        print(f"\nRaw JSON reply (first 400 characters):\n{str(full['Belief_Raw'])[:400]}")
    print()

for i, (title, idx) in enumerate(examples.items()):
    show_example(title, idx, show_raw=i == 0)